In [43]:
import pandas as pd
from pathlib import Path
from sklearn.impute import KNNImputer
import numpy as np
from posixpath import sep

In [34]:
pasta_raw = Path('../data/weather/raw')
pasta_clean = Path('../data/weather/clean')

In [19]:
df_dados_estacoes = pd.read_csv('../data/weather/dados_estacao.csv')

df_dados_estacoes.head(5)

df_dados_estacoes = df_dados_estacoes.drop(columns=['nome_estacao','latitude', 'longitude',
       'altitude', 'data_inicio_operacao', 'tipo_estacao',
       'entidade_responsavel', 'eh_capital', 'codigo_distrito', 'codigo_oscar',
       'codigo_wsi'])

In [28]:
df_operante = df_dados_estacoes[df_dados_estacoes['situacao_status']=='Operante']
df_operante.head 

<bound method NDFrame.head of     codigo_estacao estado_uf situacao_status
1             A472        BA        Operante
3             B828        RS        Operante
4             A657        ES        Operante
6             A756        MS        Operante
7             S733        MS        Operante
..             ...       ...             ...
672           A634        ES        Operante
673           A938        RO        Operante
674           A612        ES        Operante
675           A414        BA        Operante
678           A255        MA        Operante

[496 rows x 3 columns]>

In [ ]:
if 'df_operante' not in locals() and 'df_operante' not in globals():
    print("Variável 'df_operante' não encontrada. Carregando dados_estacao.csv...")
    df_estacoes = pd.read_csv('../data/weather/dados_estacao.csv')
    df_operante = df_estacoes[df_estacoes['situacao_status'] == 'Operante']

print(f"Total de estações operantes para processar: {len(df_operante)}")

imputer = KNNImputer(n_neighbors=3)

sucessos = 0
pulados = 0
imputador = 0

for idx, row in df_operante.iterrows():
    codigo = row['codigo_estacao']
    estado = row['estado_uf']
    
    arquivos_encontrados = list(pasta_raw.glob(f"dados_{codigo}_*.csv"))
    
    if not arquivos_encontrados:
        pulados += 1
        continue
        
    caminho_arquivo = arquivos_encontrados[0]
    
    try:
        df_raw = pd.read_csv(
            caminho_arquivo, 
            skiprows=10, 
            sep=';', 
            decimal=',', 
            na_values=['null', '']
        )
        
        df_raw = df_raw.loc[:, ~df_raw.columns.str.contains('^Unnamed')]
        df_raw = df_raw.dropna(how='all', axis=1)
        
        if df_raw.empty:
            continue
            
        coluna_data = 'Data Medicao'
        colunas_numericas = [col for col in df_raw.columns if col != coluna_data]
        
        for col in colunas_numericas:
            df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')
            
            if df_raw[col].isna().all():
                df_raw[col] = df_raw[col].fillna(0)
        
        if df_raw[colunas_numericas].isna().any().any():
            df_imputed = imputer.fit_transform(df_raw[colunas_numericas])
            df_raw[colunas_numericas] = df_imputed
            imputador += 1
            
        pasta_destino = pasta_clean / str(estado).upper()
        pasta_destino.mkdir(parents=True, exist_ok=True)
        
        caminho_salvamento = pasta_destino / f"dados_{codigo}_clean.csv"
        df_raw.to_csv(caminho_salvamento, index=False, sep=',', decimal='.')
        
        sucessos += 1
        
    except Exception as e:
        print(f"Erro ao processar estação {codigo}: {e}")

print(f"Estações limpas e salvas com sucesso: {sucessos}")
print(f"Estações sem arquivo correspondente: {pulados}")
print(f"Imputações: {imputador}")



Total de estações operantes para processar: 496

--- Pipeline Finalizada ---
Estações limpas e salvas com sucesso: 453
Estações sem arquivo correspondente: 43
Imputações: 434
